In [3]:
import pandas as pd
import sqlite3

# Load dataset
df = pd.read_csv('shop_dataset.csv')

# Connect to SQLite in-memory database
conn = sqlite3.connect(':memory:')
df.to_sql('pos_transactions', conn, index=False)

# Intermediate SQL Query: Payment Method Breakdown
payment_query = """
    SELECT 
        Transaction_Type,
        COUNT(CustomerID) AS Total_Transactions,
        SUM(Price) AS Total_Revenue,
        ROUND(AVG(Price), 2) AS Average_Transaction_Value
    FROM pos_transactions
    GROUP BY Transaction_Type
    ORDER BY Total_Revenue DESC;
"""
print("--- Payment Method Insights ---")
print(pd.read_sql_query(payment_query, conn))

--- Payment Method Insights ---
  Transaction_Type  Total_Transactions  Total_Revenue  \
0             Card                  14           8095   
1             Cash                  16           7509   
2      Net Banking                  11           5249   
3              UPI                   9           4717   

   Average_Transaction_Value  
0                     578.21  
1                     469.31  
2                     477.18  
3                     524.11  


In [4]:
monthly_trend_query = """
    SELECT 
        SUBSTR(Date, 4, 2) AS Month, -- extracting MM from DD-MM-YYYY format
        COUNT(CustomerID) AS Monthly_Orders,
        SUM(Price) AS Monthly_Revenue
    FROM pos_transactions
    GROUP BY Month
    ORDER BY Month ASC;
"""
print("\n--- Monthly Revenue Trends ---")
print(pd.read_sql_query(monthly_trend_query, conn))


--- Monthly Revenue Trends ---
  Month  Monthly_Orders  Monthly_Revenue
0    01              10             3507
1    02               7             3753
2    03               8             3860
3    04               8             4421
4    05               5             4165
5    06              12             5864


In [5]:
demographic_query = """
    WITH CustomerSpending AS (
        SELECT 
            CustomerID,
            Name,
            Sex,
            Age,
            SUM(Price) AS Total_Spent,
            COUNT(Product_ID) AS Items_Bought
        FROM pos_transactions
        GROUP BY CustomerID, Name, Sex, Age
    ),
    RankedCustomers AS (
        SELECT 
            Name,
            Sex,
            Age,
            Total_Spent,
            DENSE_RANK() OVER (ORDER BY Total_Spent DESC) as Spending_Rank
        FROM CustomerSpending
    )
    SELECT * FROM RankedCustomers
    WHERE Spending_Rank <= 5;
"""
print("\n--- Top High-Value Customers (Window Function & CTE) ---")
print(pd.read_sql_query(demographic_query, conn))


--- Top High-Value Customers (Window Function & CTE) ---
          Name     Sex  Age  Total_Spent  Spending_Rank
0    Zara Shah    Male   28         1247              1
1   Ritu Joshi  Female   60         1230              2
2  Priya Surve  Female   39         1200              3
3    Usha Shah    Male   60         1194              4
4  Priya Desai  Female   20         1188              5
